# TCGA-BRCA Indexed Clinical vs Local Treatment Review V1

This notebook is review-only. It reads the latest saved indexed-clinical-vs-local-treatment outputs from disk,
re-exports review TSVs into `05-results`, and does not rerun the comparison workflow, mutate raw downloads,
build treatment arms, or perform modeling.


In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display


def detect_repo_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Unable to locate the repository root from the notebook path.')


def read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)


repo_root = detect_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'source'
    / 'tcga_brca_indexed_clinical_vs_local_treatment_v1_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest indexed clinical vs local treatment pointer not found: {latest_pointer_path}'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
run_log_path = repo_root / latest_pointer['run_log_json']
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

patient_level_df = read_tsv(repo_root / latest_pointer['indexed_clinical_treatment_patient_level_v1_tsv'])
overlap_df = read_tsv(repo_root / latest_pointer['indexed_vs_local_treatment_overlap_v1_tsv'])
gap_df = read_tsv(repo_root / latest_pointer['indexed_vs_local_treatment_gap_resolution_v1_tsv'])
summary_df = read_tsv(repo_root / latest_pointer['indexed_vs_local_treatment_summary_v1_tsv'])

results_root = repo_root / '09-trials' / '01-tcga-only-source-audited' / '05-results'
results_root.mkdir(parents=True, exist_ok=True)


In [ ]:
review_patient_level_path = results_root / '136_indexed_clinical_treatment_patient_level_v1.tsv'
review_overlap_path = results_root / '137_indexed_vs_local_treatment_overlap_v1.tsv'
review_gap_path = results_root / '138_indexed_vs_local_treatment_gap_resolution_v1.tsv'
review_summary_path = results_root / '139_indexed_vs_local_treatment_summary_v1.tsv'

patient_level_df.to_csv(review_patient_level_path, sep='\t', index=False)
overlap_df.to_csv(review_overlap_path, sep='\t', index=False)
gap_df.to_csv(review_gap_path, sep='\t', index=False)
summary_df.to_csv(review_summary_path, sep='\t', index=False)

print(f'Saved: {review_patient_level_path}')
print(f'Saved: {review_overlap_path}')
print(f'Saved: {review_gap_path}')
print(f'Saved: {review_summary_path}')


In [ ]:
print('=== Latest pointer ===')
display(pd.DataFrame([latest_pointer]))

validation_df = pd.DataFrame(
    [{'check': key, 'value': str(value)} for key, value in run_log.get('validation', {}).items()]
)
print('\n=== Validation ===')
display(validation_df)

counts_df = pd.DataFrame(
    [{'metric': key, 'value': str(value)} for key, value in run_log.get('counts', {}).items()]
)
print('\n=== Key counts ===')
display(counts_df)

print('\n=== Direct answers ===')
display(summary_df[summary_df['summary_section'].isin(['questions', 'decision'])].reset_index(drop=True))


In [ ]:
print('=== Overlap ===')
display(overlap_df)

print('\n=== Gap resolution ===')
display(gap_df)


In [ ]:
focus_df = patient_level_df[patient_level_df['cohort_scope'] == 'local_profile'].copy()

no_local_drug_df = focus_df[focus_df['local_profile_has_any_drug_row'] == 'no']
no_local_any_df = focus_df[
    (focus_df['local_profile_has_any_drug_row'] == 'no')
    & (focus_df['local_profile_has_any_radiation_row'] == 'no')
]

print('=== No-local-drug incremental classes ===')
display(
    no_local_drug_df.groupby('indexed_incremental_value_class', as_index=False)
    .size()
    .rename(columns={'size': 'patient_count'})
    .sort_values(['patient_count', 'indexed_incremental_value_class'], ascending=[False, True])
    .reset_index(drop=True)
)

print('\n=== No-local-drug-or-radiation incremental classes ===')
display(
    no_local_any_df.groupby('indexed_incremental_value_class', as_index=False)
    .size()
    .rename(columns={'size': 'patient_count'})
    .sort_values(['patient_count', 'indexed_incremental_value_class'], ascending=[False, True])
    .reset_index(drop=True)
)

indexed_only_df = patient_level_df[patient_level_df['cohort_scope'] == 'indexed_only'].reset_index(drop=True)
print('\n=== Indexed-only cases not in the current local profile ===')
display(indexed_only_df[['bcr_patient_barcode', 'indexed_case_id', 'indexed_treatment_type_values_json', 'indexed_incremental_value_class']])
